**Note: Run servers on local instead of Colab**





In [1]:
pip install --quiet flask flask-ngrok scikit-learn joblib


Note: you may need to restart the kernel to use updated packages.


In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
import joblib

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])
pipeline.fit(X_train, y_train)
joblib.dump(pipeline, 'model.joblib')


['model.joblib']

In [2]:
%%writefile app.py
from flask import Flask, request, jsonify
from flask_ngrok import run_with_ngrok
import joblib

app = Flask(__name__)
run_with_ngrok(app)
model = joblib.load('model.joblib')

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json()
    features = data['features']
    pred = model.predict([features])[0]
    return jsonify({'prediction': int(pred)})

@app.route('/health', methods=['GET'])
def health():
    return 'OK', 200

if __name__ == '__main__':
    app.run()


Writing app.py


In [4]:
!python app.py


^C


In [5]:
import requests
payload = {'features': X_test[0].tolist()}
print(requests.post('http://127.0.0.1:5000/predict', json=payload).json())


{'prediction': 1}



## Exercise: Model Serialization & Deployment

### 1. Data Preparation  
- Load the Breast Cancer Wisconsin dataset with `sklearn.datasets.load_breast_cancer`  ([Save and Load Machine Learning Models in Python with scikit-learn](https://www.geeksforgeeks.org/save-and-load-machine-learning-models-in-python-with-scikit-learn/?utm_source=chatgpt.com)).  
- Split into train/test (80/20) and standardize features with `StandardScaler`  ([Save and Load Machine Learning Models in Python with scikit-learn](https://www.geeksforgeeks.org/save-and-load-machine-learning-models-in-python-with-scikit-learn/?utm_source=chatgpt.com)).  

### 2. Pipeline Construction  
- Build a scikit-learn `Pipeline` combining `StandardScaler` and `RandomForestClassifier(n_estimators=100, random_state=42)`  ([Model Serialization using pickle and joblib - Kaggle](https://www.kaggle.com/code/tasnimniger/model-serialization-using-pickle-and-joblib?utm_source=chatgpt.com)).  
- Fit the pipeline on the training data.  

### 3. Serialization  
- Save the fitted pipeline as `model_v1.pkl` using `pickle.dump(..., protocol=5)`  ([Save Machine Learning Model Using Pickle and Joblib](https://www.analyticsvidhya.com/blog/2021/08/quick-hacks-to-save-machine-learning-model-using-pickle-and-joblib/?utm_source=chatgpt.com)).  
- Save the same pipeline as `model_v1.joblib` with `joblib.dump(...)`  ([STEP 2: Model serialization and pickling | AI Planet (formerly DPhi)](https://aiplanet.com/learn/machine-learning-bootcamp/module-6-model-deployment/840/step-2-model-serialization-and-pickling?utm_source=chatgpt.com)).  

### 4. Version Control & Metadata  
- Retrain the pipeline twice more with different `random_state` values (e.g., 24, 2025) and save as `model_v2.[pkl|joblib]`, `model_v3.[pkl|joblib]`.  
- Create `model_metadata.json` capturing for each version: `{version, filename, saved_at}` in ISO format  ([Save and Load Machine Learning Models in Python with scikit-learn](https://machinelearningmastery.com/save-load-machine-learning-models-python-scikit-learn/?utm_source=chatgpt.com)).  

### 5. Deserialization & Validation  
- Load each Pickle and Joblib file, run `.predict()` on the standardized test set, and compute accuracy with `sklearn.metrics.accuracy_score`.  
- Verify that all versions produce expected accuracy and identical predictions across formats  ([Save Machine Learning Model Using Pickle and Joblib](https://www.analyticsvidhya.com/blog/2021/08/quick-hacks-to-save-machine-learning-model-using-pickle-and-joblib/?utm_source=chatgpt.com)).  

### 6. Benchmarking  
- Measure and compare file sizes (`os.path.getsize`) and load times (`time.time()` deltas) for Pickle vs Joblib  ([Use Of Pickle & Joblib To Dump And Load Machine Learning Model](https://www.youtube.com/watch?v=N4HO1HDuK4o&utm_source=chatgpt.com)).  
- Plot or tabulate the results in your notebook.  

### 7. (Bonus) Minimal REST API  
- Implement a Flask app (`app.py`) exposing two endpoints:  
  - `GET /models` → returns `model_metadata.json`.  
  - `POST /predict` → accepts JSON `{"features": […]}`, loads the **latest** model version, and returns `{"prediction": <class>}`  ([How to Save Trained Model in Python - Neptune.ai](https://neptune.ai/blog/saving-trained-model-in-python?utm_source=chatgpt.com)).  
- Demonstrate usage via Python’s `requests` in Colab.  

### Deliverables  
1. A Colab notebook with all code cells.  
2. Saved model files (`.pkl`, `.joblib`) and `model_metadata.json`.  
3. A brief markdown report comparing Pickle vs Joblib (size, speed).  
4. (Optional) Flask app code and sample requests.

In [8]:
# Step 1: Data Preparation
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load dataset
X, y = load_breast_cancer(return_X_y=True)

# Split into train and test sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train shape: {X_train_scaled.shape}")
print(f"Test shape: {X_test_scaled.shape}")



Train shape: (455, 30)
Test shape: (114, 30)


In [9]:
# Step 2: Pipeline Construction
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# Build the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Standardize features
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))  # Classifier
])

# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

# Evaluate on the test set
test_accuracy = pipeline.score(X_test, y_test)
print(f"Test Accuracy: {test_accuracy:.4f}")

Test Accuracy: 0.9649


In [10]:
# Step 3: Serialization
import pickle
import joblib

# Save the pipeline as model_v1.pkl using pickle (protocol 5)
with open('model_v1.pkl', 'wb') as f:
    pickle.dump(pipeline, f, protocol=5)

# Save the pipeline as model_v1.joblib using joblib
joblib.dump(pipeline, 'model_v1.joblib')

print("Models saved as model_v1.pkl and model_v1.joblib")


Models saved as model_v1.pkl and model_v1.joblib


In [11]:
# Step 4: Version Control & Metadata
import json
from datetime import datetime

model_versions = [
    {"version": "v1", "random_state": 42},
    {"version": "v2", "random_state": 24},
    {"version": "v3", "random_state": 2025},
]

metadata = []

for v in model_versions:
    # Create and train pipeline
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=v["random_state"]))
    ])
    pipe.fit(X_train, y_train)
    
    # Filenames
    pkl_filename = f"model_{v['version']}.pkl"
    joblib_filename = f"model_{v['version']}.joblib"
    
    # Save with pickle
    with open(pkl_filename, "wb") as f:
        pickle.dump(pipe, f, protocol=5)
    # Save with joblib
    joblib.dump(pipe, joblib_filename)
    
    # Metadata entry for each format
    now = datetime.now().isoformat()
    metadata.append({"version": v["version"], "filename": pkl_filename, "saved_at": now})
    metadata.append({"version": v["version"], "filename": joblib_filename, "saved_at": now})

# Save metadata to JSON
with open("model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("All model versions saved and metadata written to model_metadata.json")

All model versions saved and metadata written to model_metadata.json


In [12]:
# Step 5: Deserialization & Validation
from sklearn.metrics import accuracy_score
import numpy as np

model_files = [
    ("model_v1.pkl", "model_v1.joblib"),
    ("model_v2.pkl", "model_v2.joblib"),
    ("model_v3.pkl", "model_v3.joblib"),
]

for pkl_file, joblib_file in model_files:
    # Load with pickle
    with open(pkl_file, "rb") as f:
        model_pickle = pickle.load(f)
    # Load with joblib
    model_joblib = joblib.load(joblib_file)
    
    # Predict
    pred_pickle = model_pickle.predict(X_test)
    pred_joblib = model_joblib.predict(X_test)
    
    # Accuracy
    acc_pickle = accuracy_score(y_test, pred_pickle)
    acc_joblib = accuracy_score(y_test, pred_joblib)
    
    # Check identical predictions
    identical = np.array_equal(pred_pickle, pred_joblib)
    
    print(f"{pkl_file} vs {joblib_file}:")
    print(f"  Pickle accuracy: {acc_pickle:.4f}")
    print(f"  Joblib accuracy: {acc_joblib:.4f}")
    print(f"  Predictions identical: {identical}\n")

model_v1.pkl vs model_v1.joblib:
  Pickle accuracy: 0.9649
  Joblib accuracy: 0.9649
  Predictions identical: True

model_v2.pkl vs model_v2.joblib:
  Pickle accuracy: 0.9649
  Joblib accuracy: 0.9649
  Predictions identical: True

model_v3.pkl vs model_v3.joblib:
  Pickle accuracy: 0.9561
  Joblib accuracy: 0.9561
  Predictions identical: True



In [13]:
# Step 6: Benchmarking
import os
import time

benchmark_results = []

for version in ["v1", "v2", "v3"]:
    pkl_file = f"model_{version}.pkl"
    joblib_file = f"model_{version}.joblib"
    
    # File sizes
    size_pkl = os.path.getsize(pkl_file)
    size_joblib = os.path.getsize(joblib_file)
    
    # Load times
    start = time.time()
    with open(pkl_file, "rb") as f:
        _ = pickle.load(f)
    load_time_pkl = time.time() - start
    
    start = time.time()
    _ = joblib.load(joblib_file)
    load_time_joblib = time.time() - start
    
    benchmark_results.append({
        "version": version,
        "pkl_size": size_pkl,
        "joblib_size": size_joblib,
        "pkl_load_time": load_time_pkl,
        "joblib_load_time": load_time_joblib
    })

# Print results as a table
print(f"{'Version':<8} {'Pickle Size (KB)':<18} {'Joblib Size (KB)':<18} {'Pickle Load (s)':<16} {'Joblib Load (s)':<16}")
for res in benchmark_results:
    print(f"{res['version']:<8} {res['pkl_size']/1024:<18.2f} {res['joblib_size']/1024:<18.2f} {res['pkl_load_time']:<16.5f} {res['joblib_load_time']:<16.5f}")

Version  Pickle Size (KB)   Joblib Size (KB)   Pickle Load (s)  Joblib Load (s) 
v1       308.25             318.78             0.00600          0.03205         
v2       314.82             325.35             0.00322          0.02497         
v3       316.38             326.91             0.00324          0.02557         


In [14]:
from flask import Flask, request, jsonify
import joblib
import json
import os

app = Flask(__name__)

# Helper to get latest model (by version string, e.g., v3 > v2 > v1)
def get_latest_model_filename():
    with open("model_metadata.json", "r") as f:
        metadata = json.load(f)
    # Only consider joblib files for prediction
    joblib_models = [m for m in metadata if m["filename"].endswith(".joblib")]
    # Sort by version (assuming v1, v2, v3, ...)
    latest = sorted(joblib_models, key=lambda x: x["version"], reverse=True)[0]
    return latest["filename"]

@app.route("/models", methods=["GET"])
def models():
    with open("model_metadata.json", "r") as f:
        metadata = json.load(f)
    return jsonify(metadata)

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json()
    features = data.get("features")
    if features is None:
        return jsonify({"error": "Missing 'features' in request"}), 400
    model_filename = get_latest_model_filename()
    model = joblib.load(model_filename)
    pred = model.predict([features])[0]
    return jsonify({"prediction": int(pred)})

if __name__ == "__main__":
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (windowsapi)


SystemExit: 1

c:\Users\Lenovo\miniconda3\envs\python_ml\Lib\site-packages\IPython\core\interactiveshell.py:3680: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
import requests
import numpy as np

# Use a sample from your test set
sample = X_test[0].tolist()
response = requests.post("http://127.0.0.1:5000/predict", json={"features": sample})
print(response.json())

{'prediction': 1}


In [15]:
!python app.py

^C


In [16]:
import requests
import numpy as np

# Use a sample from your test set
sample = X_test[0].tolist()
response = requests.post("http://127.0.0.1:5000/predict", json={"features": sample})
print(response.json())

{'prediction': 1}


In [ ]:
import requests
import numpy as np

# Use a sample from your test set
sample = X_test[0].tolist()
response = requests.post("http://127.0.0.1:5000/predict", json={"features": sample})
print(response.json())

{'prediction': 1}
